# PetFinder Adoption — бинарная классификация / binary classification

**RU.** Предсказание, будет ли животное из приюта (кошка или собака) взято в новую семью достаточно быстро. Исходная задача Kaggle *PetFinder.my Adoption Prediction* 5-классовая; мы сводим её к бинарной: классы 0–2 (усыновили в течение ≈месяца) → `0`, 3–4 (долго или не усыновили) → `1`. В работе: предобработка числовых и категориальных признаков, две стратегии скейлинга для разных байесовских моделей, `GridSearchCV` для `BernoulliNB`, `MultinomialNB` и `KNN`, сравнение моделей по метрикам `Precision` / `Recall` / `F1` / `ROC-AUC`.

**EN.** Predicting whether a shelter animal (cat or dog) will be adopted reasonably quickly. The original Kaggle *PetFinder.my Adoption Prediction* task has 5 classes; we reduce it to a binary problem: classes 0–2 (adopted within ≈a month) → `0`, 3–4 (slow or never adopted) → `1`. The project covers: preprocessing of numeric and categorical features, two scaling strategies for different Naive-Bayes variants, `GridSearchCV` for `BernoulliNB`, `MultinomialNB` and `KNN`, and a head-to-head comparison using `Precision` / `Recall` / `F1` / `ROC-AUC`.

**Dataset / датасет:** <https://www.kaggle.com/c/petfinder-adoption-prediction>

---

### Задачи / Tasks

- **RU.** Выполнить предобработку признаков и построить оптимальные модели `kNN` и все типы классификаторов Байеса. Для подбора гиперпараметров использовать `GridSearchCV`.
- **EN.** Preprocess the features and train the best possible `kNN` and Naive-Bayes classifiers. Use `GridSearchCV` for hyperparameter tuning.

## 1. Импорты / Imports

In [ ]:
import numpy as np
import pandas as pd
import requests

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.naive_bayes import BernoulliNB, MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler, RobustScaler

RANDOM_STATE = 42
pd.set_option("display.max_columns", 100)

## 2. Загрузка данных / Data loading

**RU.** Скачиваем основной `train.csv` (зеркало на Google Sheets). В ноутбуке используется только он; тестовый файл без меток и справочники пород / цветов / штатов нужны для полноценной подачи в Kaggle и в этот учебный пайплайн не включены.

**EN.** Download only `train.csv` (a Google-Sheets mirror). The unlabelled test file and the breed / colour / state lookup tables are only required for a Kaggle submission and are not used in this study pipeline.

In [ ]:
DATA_URL = (
    "https://docs.google.com/spreadsheets/d/"
    "1LQRnseu8pF1AbN4ZPnYUTtfGsAZ8AiDbM4GOVNUBWVs/export?format=csv"
)
DATA_FILE = "train.csv"


def download_dataset(url: str, file_name: str) -> None:
    """Скачать CSV в file_name, если файл ещё не загружен.

    Download CSV into ``file_name`` only if it is not already
    present on disk.
    """
    from os.path import exists

    if exists(file_name):
        print(f"{file_name} уже загружен / already downloaded")
        return
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    with open(file_name, "wb") as f:
        f.write(response.content)
    print(f"Файл сохранён / file saved: {file_name}")


download_dataset(DATA_URL, DATA_FILE)
df = pd.read_csv(DATA_FILE)
df.head()

,Type,Name,Age,Breed1,Breed2,Gender,Color1,Color2,Color3,MaturitySize,FurLength,Vaccinated,Dewormed,Sterilized,Health,Quantity,Fee,State,RescuerID,VideoAmt,Description,PetID,PhotoAmt,AdoptionSpeed
0,2,Nibble,3,299,0,1,1,7,0,1,1,2,2,2,1,1,100,41326,8480853f516546f6cf33aa88cd76c379,0,Nibble is a 3+ month old ball of cuteness. He ...,86e1089a3,1.0,2
1,2,No Name Yet,1,265,0,1,1,2,0,2,2,3,3,3,1,1,0,41401,3082c7125d8fb66f7dd4bff4192c8b14,0,I just found it alone yesterday near my apartm...,6296e909a,2.0,0
2,1,Brisco,1,307,0,1,2,7,0,2,2,1,1,2,1,1,0,41326,fa90fa5b1ee11c86938398b60abc32cb,0,Their pregnant mother was dumped by her irresp...,3422e4906,7.0,3
3,1,Miko,4,307,0,2,1,2,0,2,1,1,1,2,1,1,150,41401,9238e4f44c71a75282e62f7136c6b240,0,"Good guard dog, very alert, active, obedience ...",5842f1ff5,8.0,2
4,1,Hunter,1,307,0,1,1,0,0,2,1,2,2,2,1,1,0,41326,95481e953f8aed9ec3d16fc4509537e8,0,This handsome yet cute boy is up for adoption....,850a43f90,3.0,2


## 3. Целевая переменная / Target engineering

**RU.** Исходная колонка `AdoptionSpeed` принимает значения 0–4 (чем меньше, тем быстрее усыновили). Сводим задачу к бинарной: 0–2 (быстро) → `0`, 3–4 (медленно / не усыновили) → `1`.

**EN.** The original `AdoptionSpeed` column has values 0–4 (smaller = faster adoption). We collapse it to binary: 0–2 (fast) → `0`, 3–4 (slow / never) → `1`.

In [ ]:
print(df["AdoptionSpeed"].value_counts().sort_index())

,count
AdoptionSpeed,
4,4197
2,4037
3,3259
1,3090
0,410


In [ ]:
SPEED_TO_BINARY = {0: 0, 1: 0, 2: 0, 3: 1, 4: 1}
df["AdoptionSpeed_binary"] = df["AdoptionSpeed"].map(SPEED_TO_BINARY)
print(df["AdoptionSpeed_binary"].value_counts())

,count
AdoptionSpeed_binary,
0,7537
1,7456


## 4. Отбор признаков / Feature selection

**RU.** Удаляем идентификаторы, свободный текст и служебные колонки, которые не несут сигнала для этой задачи.

**EN.** Drop identifiers, free-form text and bookkeeping columns that carry no signal for this task.

In [ ]:
DROP_COLUMNS = [
    "Type",           # species — dog/cat; dropped for simplicity
    "Name",
    "RescuerID",
    "VideoAmt",
    "PetID",
    "PhotoAmt",
    "Description",
    "AdoptionSpeed",
]

features = df.drop(columns=DROP_COLUMNS)
y = features.pop("AdoptionSpeed_binary")

print(features.info())
print("\nAny NaNs? / есть ли пропуски? ", features.isnull().values.any())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14993 entries, 0 to 14992
Data columns (total 27 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Unnamed: 0                  14993 non-null  int64  
 1   Type                        14993 non-null  int64  
 2   Name                        13728 non-null  object 
 3   Age                         14993 non-null  int64  
 4   Breed1                      14993 non-null  int64  
 5   Breed2                      14993 non-null  int64  
 6   Gender                      14993 non-null  int64  
 7   Color1                      14993 non-null  int64  
 8   Color2                      14993 non-null  int64  
 9   Color3                      14993 non-null  int64  
 10  MaturitySize                14993 non-null  int64  
 11  FurLength                   14993 non-null  int64  
 12  Vaccinated                  14993 non-null  int64  
 13  Dewormed                    149

## 5. Разделение числовых и категориальных / Numeric vs. categorical split

In [ ]:
NUMERIC_FEATURES = ["Age", "Fee", "Quantity"]
CATEGORICAL_FEATURES = [
    "Breed1",
    "Breed2",
    "Gender",
    "Color1",
    "Color2",
    "Color3",
    "State",
    "MaturitySize",
    "FurLength",
    "Vaccinated",
    "Dewormed",
    "Sterilized",
    "Health",
]

## 6. Два варианта масштабирования / Two scaling strategies

**RU.** `BernoulliNB` и `KNN` дружат с признаками, центрированными по медиане — берём `RobustScaler`. `MultinomialNB` требует неотрицательных значений, поэтому для него: сначала клипируем `Fee` по 1–99%-квантилям (чтобы убрать выбросы), затем `MinMaxScaler` → признаки в диапазоне `[0, 1]`.

**EN.** `BernoulliNB` and `KNN` work well with median-centred features → use `RobustScaler`. `MultinomialNB` needs non-negative inputs, so for it we first clip `Fee` to the 1–99 % quantile range (to suppress outliers) and then apply `MinMaxScaler`, which keeps values in `[0, 1]`.

In [ ]:
# Vector 1: RobustScaler for BernoulliNB / KNN
robust_scaler = RobustScaler()
x_num_robust = pd.DataFrame(
    robust_scaler.fit_transform(features[NUMERIC_FEATURES]),
    columns=NUMERIC_FEATURES,
    index=features.index,
)

# Vector 2: MinMaxScaler + Fee clipping for MultinomialNB
x_num_mnb = features[NUMERIC_FEATURES].copy()
x_num_mnb["Fee"] = x_num_mnb["Fee"].clip(
    lower=x_num_mnb["Fee"].quantile(0.01),
    upper=x_num_mnb["Fee"].quantile(0.99),
)
minmax_scaler = MinMaxScaler()
x_num_mnb[NUMERIC_FEATURES] = minmax_scaler.fit_transform(x_num_mnb[NUMERIC_FEATURES])

## 7. One-hot для категориальных / One-hot for categorical

In [ ]:
x_cat = pd.get_dummies(
    features[CATEGORICAL_FEATURES],
    columns=CATEGORICAL_FEATURES,
    drop_first=False,
    dtype=int,
)

X_robust = pd.concat([x_num_robust, x_cat], axis=1)
X_mnb = pd.concat([x_num_mnb, x_cat], axis=1)

assert not X_robust.isnull().values.any(), "NaNs in X_robust"
assert not X_mnb.isnull().values.any(), "NaNs in X_mnb"

print(f"X_robust shape: {X_robust.shape}")
print(f"X_mnb    shape: {X_mnb.shape}")

## 8. Разбиение выборки / Train–validation split

In [ ]:
x_train_r, x_val_r, y_train, y_val = train_test_split(
    X_robust, y,
    test_size=0.3,
    random_state=RANDOM_STATE,
    stratify=y,
)

# Одинаковый random_state гарантирует, что y_train/y_val совпадают
# для обеих матриц признаков.
# The same random_state guarantees y_train/y_val align across both
# feature matrices.
x_train_m, x_val_m, _, _ = train_test_split(
    X_mnb, y,
    test_size=0.3,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Robust split :", x_train_r.shape, x_val_r.shape)
print("MNB    split :", x_train_m.shape, x_val_m.shape)

## 9. Подбор гиперпараметров `GridSearchCV` / Hyperparameter tuning

**RU.** GaussianNB не включён: он плохо работает с one-hot признаками, где распределение бинарно.

**EN.** GaussianNB is excluded: it performs poorly on one-hot features, whose distribution is binary, not Gaussian.

In [ ]:
search_specs = {
    "BernoulliNB": (
        BernoulliNB(),
        {"alpha": [0.01, 0.1, 0.5, 1.0, 2.0, 2.5, 3.0], "binarize": [None, 0.0]},
        (x_train_r, y_train),
    ),
    "MultinomialNB": (
        MultinomialNB(),
        {"alpha": [0.01, 0.1, 0.5, 1.0, 2.0, 3.0, 4.0, 5.0]},
        (x_train_m, y_train),
    ),
    "KNN": (
        KNeighborsClassifier(),
        {"n_neighbors": list(range(3, 32, 2))},
        (x_train_r, y_train),
    ),
}

best_params = {}
for name, (estimator, param_grid, (x_tr, y_tr)) in search_specs.items():
    grid = GridSearchCV(
        estimator,
        param_grid,
        scoring="roc_auc",
        cv=5,
        n_jobs=-1,
    )
    grid.fit(x_tr, y_tr)
    best_params[name] = grid.best_params_
    print(f"{name}: best_params={grid.best_params_}  best_roc_auc={grid.best_score_:.4f}")

## 10. Функция оценки / Evaluation helper

**RU.** Один helper считает `Precision`/`Recall` на нескольких порогах и финальные метрики на пороге 0.5.

**EN.** A single helper reports `Precision`/`Recall` across multiple thresholds and the final metrics at the 0.5 threshold.

In [ ]:
def evaluate_classifier(
    model,
    x_val,
    y_val,
    name: str = "model",
    thresholds=(0.3, 0.4, 0.5, 0.6),
    main_threshold: float = 0.5,
) -> dict:
    """Оценить бинарный классификатор и вернуть словарь с метриками.

    Evaluate a binary classifier; print Precision/Recall for several
    thresholds and Accuracy / F1 / ROC-AUC / confusion matrix at
    ``main_threshold``. Return the headline metrics as a dict.
    """
    val_proba = model.predict_proba(x_val)[:, 1]

    print(f"=== {name} ===")
    for t in thresholds:
        pred = (val_proba >= t).astype(int)
        print(
            f"  t={t:>4}  "
            f"precision={precision_score(y_val, pred):.3f}  "
            f"recall={recall_score(y_val, pred):.3f}"
        )

    pred = (val_proba >= main_threshold).astype(int)
    metrics = {
        "model": name,
        "accuracy": accuracy_score(y_val, pred),
        "precision": precision_score(y_val, pred),
        "recall": recall_score(y_val, pred),
        "f1": f1_score(y_val, pred),
        "roc_auc": roc_auc_score(y_val, val_proba),
    }
    print(
        f"  → accuracy={metrics['accuracy']:.3f}  "
        f"precision={metrics['precision']:.3f}  "
        f"recall={metrics['recall']:.3f}  "
        f"f1={metrics['f1']:.3f}  "
        f"roc_auc={metrics['roc_auc']:.3f}"
    )
    print("  Confusion matrix:\n", confusion_matrix(y_val, pred))
    print()
    return metrics

## 11. Обучение и сравнение моделей / Training and comparison

**RU.** Модели обучаются с подобранными гиперпараметрами и оцениваются на соответствующих валидационных матрицах.

**EN.** The tuned models are refit and evaluated on their matching validation matrices.

In [ ]:
final_specs = [
    ("BernoulliNB", BernoulliNB(alpha=2.5, binarize=0.0), x_train_r, x_val_r),
    ("MultinomialNB", MultinomialNB(alpha=3.0), x_train_m, x_val_m),
    ("KNN (k=21)", KNeighborsClassifier(n_neighbors=21), x_train_r, x_val_r),
]

metrics_records = []
for name, model, x_tr, x_val in final_specs:
    model.fit(x_tr, y_train)
    metrics_records.append(evaluate_classifier(model, x_val, y_val, name=name))

pd.DataFrame(metrics_records).set_index("model").round(3)

## 12. Выводы / Conclusions

**RU.**

- `GaussianNB` неприменим к данным, где подавляющая часть признаков — one-hot (их распределение не гауссово).
- Лучше всего себя показал `BernoulliNB` — он естественно согласуется с бинарной природой one-hot. На валидации достигает `ROC-AUC ≈ 0.65`.
- `MultinomialNB` чуть уступает `BernoulliNB`, но остаётся хорошим бейзлайном при правильно нормированных неотрицательных признаках.
- `KNN` с `n_neighbors=21` чувствителен к высокой размерности после one-hot, поэтому выдаёт более низкий `ROC-AUC`.

**EN.**

- `GaussianNB` does not fit data dominated by one-hot features (their distribution is not Gaussian).
- `BernoulliNB` works best — it naturally matches the binary nature of one-hot encodings, reaching `ROC-AUC ≈ 0.65` on the validation set.
- `MultinomialNB` is slightly behind but still a solid baseline with properly scaled non-negative inputs.
- `KNN` (`n_neighbors=21`) suffers from the curse of dimensionality after one-hot expansion, hence its lower `ROC-AUC`.